# Generic Automated-vs-Manual Meta-Analysis Benchmark

This notebook compares multiple automated meta-analysis runs against manual benchmark maps from NeuroMetaBench. It is designed to work from either:
- `projects/{project_name}/analysis`
- `projects/{project_name}/{branch}/analysis`


In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import nibabel as nib
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from IPython.display import display

MANUAL_ANALYSIS_BASE = Path('/home/zorro/repos/neurometabench/analysis')
MAP_FILENAME = 'z.nii.gz'
DICE_THRESHOLD = 1.96
IMAGES_DIR = Path('images')
TABLES_DIR = Path('tables')

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)


In [ ]:
analysis_dir = Path.cwd().resolve()
if analysis_dir.name != 'analysis':
    raise RuntimeError(
        f'This notebook must be run with working directory set to an analysis folder. Current directory: {analysis_dir}'
    )

runs_root = analysis_dir.parent

if runs_root.parent.name == 'projects':
    project_root = runs_root
    project_name = runs_root.name
    placement_mode = 'project-root-analysis'
elif runs_root.parent.parent.name == 'projects':
    project_root = runs_root.parent
    project_name = runs_root.parent.name
    placement_mode = 'branched-analysis'
else:
    raise RuntimeError(
        'Could not infer project layout. Expected analysis directory under projects/{project_name}/analysis '
        'or projects/{project_name}/{branch}/analysis.'
    )

mapping_candidates = [
    runs_root / 'nmb_mappings.json',
    project_root / 'nmb_mappings.json',
]
mapping_path = next((p for p in mapping_candidates if p.exists()), None)
if mapping_path is None:
    searched = '\n'.join(str(p) for p in mapping_candidates)
    raise FileNotFoundError(f'Could not locate nmb_mappings.json. Searched:\n{searched}')

with mapping_path.open('r', encoding='utf-8') as f:
    mappings = json.load(f)

if not isinstance(mappings, dict) or not mappings:
    raise ValueError(f'Mapping file must be a non-empty JSON object: {mapping_path}')

run_dirs = sorted(
    [
        p
        for p in runs_root.iterdir()
        if p.is_dir() and (p / 'outputs' / 'meta_analysis_results').is_dir()
    ],
    key=lambda p: p.name,
)

if not run_dirs:
    raise RuntimeError(
        f'No automated run folders found under {runs_root} with outputs/meta_analysis_results.'
    )

run_names = [p.name for p in run_dirs]

print('Configuration Summary')
print('=' * 80)
print(f'analysis_dir:        {analysis_dir}')
print(f'runs_root:           {runs_root}')
print(f'project_root:        {project_root}')
print(f'project_name:        {project_name}')
print(f'placement_mode:      {placement_mode}')
print(f'mapping_path:        {mapping_path}')
print(f'manual_analysis_base:{MANUAL_ANALYSIS_BASE}')
print(f'map_filename:        {MAP_FILENAME}')
print(f'dice_threshold:      {DICE_THRESHOLD}')
print(f'discovered_runs:     {run_names}')
print(f'n_mappings:          {len(mappings)}')


In [ ]:
def manual_name_candidates(manual_name):
    candidates = [str(manual_name)]
    if str(manual_name).endswith('.txt'):
        candidates.append(str(manual_name)[:-4])

    deduped = []
    for candidate in candidates:
        if candidate and candidate not in deduped:
            deduped.append(candidate)
    return deduped

mapping_pairs = []
missing_errors = []
availability_rows = []

for manual_name, auto_name in mappings.items():
    candidates = manual_name_candidates(manual_name)

    manual_path = None
    manual_paths_checked = []
    for candidate in candidates:
        candidate_path = MANUAL_ANALYSIS_BASE / project_name / candidate / MAP_FILENAME
        manual_paths_checked.append(candidate_path)
        if candidate_path.exists():
            manual_path = candidate_path
            break

    if manual_path is None:
        checked_str = ', '.join(str(p) for p in manual_paths_checked)
        missing_errors.append(
            f'Missing manual map for mapping {manual_name} -> {auto_name}. Checked: {checked_str}'
        )

    auto_paths = {}
    missing_auto_runs = []
    for run_dir in run_dirs:
        run_name = run_dir.name
        auto_file = run_dir / 'outputs' / 'meta_analysis_results' / str(auto_name) / MAP_FILENAME
        auto_paths[run_name] = auto_file
        if not auto_file.exists():
            missing_auto_runs.append(run_name)

    if missing_auto_runs:
        missing_runs_str = ', '.join(missing_auto_runs)
        missing_errors.append(
            f'Missing automated map for mapping {manual_name} -> {auto_name} in runs: {missing_runs_str}'
        )

    mapping_pairs.append(
        {
            'manual_name': str(manual_name),
            'auto_name': str(auto_name),
            'manual_path': manual_path,
            'auto_paths': auto_paths,
        }
    )

    row = {
        'manual_name': str(manual_name),
        'auto_name': str(auto_name),
        'manual_exists': manual_path is not None,
    }
    for run_name in run_names:
        row[f'run::{run_name}'] = auto_paths[run_name].exists()
    availability_rows.append(row)

availability_df = pd.DataFrame(availability_rows)
print('\nAvailability Summary')
print('=' * 80)
display(availability_df)

if missing_errors:
    first_lines = '\n'.join(f'- {err}' for err in missing_errors)
    raise FileNotFoundError(
        'Strict map validation failed. Every mapped manual and automated output must contain '
        f'{MAP_FILENAME}.\n'
        f'{first_lines}'
    )


In [ ]:
manual_data = {}
auto_data_by_run = {run_name: {} for run_name in run_names}
shape_records = []

for pair in mapping_pairs:
    manual_name = pair['manual_name']
    auto_name = pair['auto_name']

    manual_img = nib.load(str(pair['manual_path']))
    manual_arr = manual_img.get_fdata()
    manual_data[manual_name] = manual_arr
    shape_records.append((f'manual::{manual_name}', manual_arr.shape))

    for run_name in run_names:
        auto_img = nib.load(str(pair['auto_paths'][run_name]))
        auto_arr = auto_img.get_fdata()
        auto_data_by_run[run_name][auto_name] = auto_arr
        shape_records.append((f'{run_name}::{auto_name}', auto_arr.shape))

unique_shapes = sorted({shape for _, shape in shape_records})
if len(unique_shapes) != 1:
    shape_lines = '\n'.join(f'- {name}: {shape}' for name, shape in shape_records)
    raise ValueError(
        'All maps must have identical shapes before comparison. Found mismatched shapes:\n'
        f'{shape_lines}'
    )

common_shape = unique_shapes[0]
mask = np.ones(common_shape, dtype=bool)
for arr in manual_data.values():
    mask &= np.isfinite(arr)
for run_data in auto_data_by_run.values():
    for arr in run_data.values():
        mask &= np.isfinite(arr)

n_valid_voxels = int(mask.sum())
if n_valid_voxels == 0:
    raise ValueError('No common finite voxels remained after masking.')

manual_vectors = {name: arr[mask].ravel() for name, arr in manual_data.items()}
auto_vectors_by_run = {
    run_name: {name: arr[mask].ravel() for name, arr in run_data.items()}
    for run_name, run_data in auto_data_by_run.items()
}

print('Loaded Data Summary')
print('=' * 80)
print(f'common_shape:      {common_shape}')
print(f'valid_voxels:      {n_valid_voxels}')
print(f'n_manual_maps:     {len(manual_vectors)}')
print(f'n_automated_runs:  {len(auto_vectors_by_run)}')


In [ ]:
def compute_dice(vec_a, vec_b, threshold):
    binary_a = vec_a > threshold
    binary_b = vec_b > threshold
    intersection = np.sum(binary_a & binary_b)
    volume_sum = np.sum(binary_a) + np.sum(binary_b)
    if volume_sum == 0:
        return 0.0
    return float((2.0 * intersection) / volume_sum)


def compute_pearson(vec_a, vec_b):
    if vec_a.size < 2 or vec_b.size < 2:
        return np.nan
    if np.all(vec_a == vec_a[0]) or np.all(vec_b == vec_b[0]):
        return np.nan
    return float(pearsonr(vec_a, vec_b)[0])


def sanitize_name(name):
    return re.sub(r'[^A-Za-z0-9._-]+', '_', str(name))


manual_order = [pair['manual_name'] for pair in mapping_pairs]
auto_order = [pair['auto_name'] for pair in mapping_pairs]

dice_matrices = {}
pearson_matrices = {}

for run_name in run_names:
    run_auto_vectors = auto_vectors_by_run[run_name]

    dice_df = pd.DataFrame(index=auto_order, columns=manual_order, dtype=float)
    pearson_df = pd.DataFrame(index=auto_order, columns=manual_order, dtype=float)

    for auto_name in auto_order:
        auto_vec = run_auto_vectors[auto_name]
        for manual_name in manual_order:
            manual_vec = manual_vectors[manual_name]
            dice_df.loc[auto_name, manual_name] = compute_dice(auto_vec, manual_vec, DICE_THRESHOLD)
            pearson_df.loc[auto_name, manual_name] = compute_pearson(auto_vec, manual_vec)

    dice_matrices[run_name] = dice_df
    pearson_matrices[run_name] = pearson_df

    safe_run_name = sanitize_name(run_name)
    dice_df.to_csv(TABLES_DIR / f'dice_matrix_{safe_run_name}.csv')
    pearson_df.to_csv(TABLES_DIR / f'pearson_matrix_{safe_run_name}.csv')

diag_rows = []
for run_name in run_names:
    for pair in mapping_pairs:
        auto_name = pair['auto_name']
        manual_name = pair['manual_name']
        diag_rows.append(
            {
                'run': run_name,
                'manual_name': manual_name,
                'auto_name': auto_name,
                'dice': dice_matrices[run_name].loc[auto_name, manual_name],
                'pearson_r': pearson_matrices[run_name].loc[auto_name, manual_name],
            }
        )

diag_df = pd.DataFrame(diag_rows)
diag_df['run'] = pd.Categorical(diag_df['run'], categories=run_names, ordered=True)
diag_df['auto_name'] = pd.Categorical(diag_df['auto_name'], categories=auto_order, ordered=True)
diag_df = diag_df.sort_values(['auto_name', 'run']).reset_index(drop=True)
diag_df.to_csv(TABLES_DIR / 'diagonal_metrics.csv', index=False)

summary_rows = []
for run_name in run_names:
    dice_values = dice_matrices[run_name].to_numpy().ravel()
    pearson_values = pearson_matrices[run_name].to_numpy().ravel()

    diag_dice = [dice_matrices[run_name].loc[pair['auto_name'], pair['manual_name']] for pair in mapping_pairs]
    diag_pearson = [pearson_matrices[run_name].loc[pair['auto_name'], pair['manual_name']] for pair in mapping_pairs]

    summary_rows.append(
        {
            'run': run_name,
            'n_rows': int(dice_matrices[run_name].shape[0]),
            'n_cols': int(dice_matrices[run_name].shape[1]),
            'dice_mean_full': float(np.nanmean(dice_values)),
            'dice_mean_diagonal': float(np.nanmean(diag_dice)),
            'pearson_mean_full': float(np.nanmean(pearson_values)),
            'pearson_mean_diagonal': float(np.nanmean(diag_pearson)),
        }
    )

summary_df = pd.DataFrame(summary_rows).set_index('run')
summary_df.to_csv(TABLES_DIR / 'run_summary.csv')

print('Diagonal Dice (auto_name x run)')
display(diag_df.pivot(index='auto_name', columns='run', values='dice').round(3))

print('Diagonal Pearson r (auto_name x run)')
display(diag_df.pivot(index='auto_name', columns='run', values='pearson_r').round(3))

print('Per-run summary')
display(summary_df.round(3))


In [ ]:
for run_name in run_names:
    safe_run_name = sanitize_name(run_name)

    fig, ax = plt.subplots(figsize=(max(8, 1.4 * len(manual_order)), max(6, 1.1 * len(auto_order))))
    sns.heatmap(
        dice_matrices[run_name],
        annot=True,
        fmt='.3f',
        cmap='viridis',
        vmin=0,
        vmax=1,
        ax=ax,
        cbar_kws={'label': 'Dice coefficient'},
    )
    ax.set_title(f'Dice Matrix: {run_name} (Automated vs Manual)', fontweight='bold')
    ax.set_xlabel('Manual benchmark annotation')
    ax.set_ylabel('Automated annotation')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / f'dice_heatmap_{safe_run_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

for run_name in run_names:
    safe_run_name = sanitize_name(run_name)

    fig, ax = plt.subplots(figsize=(max(8, 1.4 * len(manual_order)), max(6, 1.1 * len(auto_order))))
    sns.heatmap(
        pearson_matrices[run_name],
        annot=True,
        fmt='.3f',
        cmap='coolwarm',
        vmin=-1,
        vmax=1,
        center=0,
        ax=ax,
        cbar_kws={'label': 'Pearson r'},
    )
    ax.set_title(f'Pearson Matrix: {run_name} (Automated vs Manual)', fontweight='bold')
    ax.set_xlabel('Manual benchmark annotation')
    ax.set_ylabel('Automated annotation')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / f'pearson_heatmap_{safe_run_name}.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
plot_df = diag_df.copy()

fig, ax = plt.subplots(figsize=(max(12, 1.6 * len(auto_order)), 6))
sns.barplot(
    data=plot_df,
    x='auto_name',
    y='dice',
    hue='run',
    order=auto_order,
    hue_order=run_names,
    errorbar=None,
    ax=ax,
)
ax.set_title(f'Diagonal Dice (z > {DICE_THRESHOLD}) by Run', fontweight='bold')
ax.set_xlabel('Automated annotation (mapped to manual)')
ax.set_ylabel('Dice coefficient')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Run', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'dice_diagonal_grouped.png', dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(max(12, 1.6 * len(auto_order)), 6))
sns.barplot(
    data=plot_df,
    x='auto_name',
    y='pearson_r',
    hue='run',
    order=auto_order,
    hue_order=run_names,
    errorbar=None,
    ax=ax,
)
ax.set_title('Diagonal Pearson r by Run', fontweight='bold')
ax.set_xlabel('Automated annotation (mapped to manual)')
ax.set_ylabel('Pearson r')
ax.set_ylim(-1, 1)
ax.axhline(0, color='black', linewidth=1, alpha=0.6)
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Run', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'pearson_diagonal_grouped.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved image outputs to:', IMAGES_DIR.resolve())
print('Saved table outputs to:', TABLES_DIR.resolve())
